# Optimizers in Pytorch

In this notebook we are going to pay special attention to optimizers in Pytorch.

Optimizers are grouped within the torch.optim package. Their role within the learning process is to integrate with the tensor autograd mechanism and provide an implementation of the backward pass. To do this, they need to obtain information about how far the model is from the desired output in order to calculate the gradients to apply to the model parameters. Loss functions are treated separately in another notebook.

Some common features of Pytorch optimizers are:

- Maintain the current state and update parameters using a calculation of their gradients
- They have a common interface that makes it easy
- That they are easily interchangeable
- Ad-hoc optimizers can be implemented
- They receive an iterable with the learnable parameters of the model
- They also receive other parameters, called optimizer hyperparameters, which allow their configuration (e.g. learning rate, momentum)

Main methods:

- *backward()*: Calculates gradients. Applies to the loss function, not the optimizer.
- *zero_grad()*: Sets the optimizer gradients to 0
- *step()*: Applies the gradients calculated by *backward()* to the model parameters

The base class of the optimizer is optim.Optimizer and it receives as parameters:
- An interactive containing the parameters to be optimized
- A dictionary with the optimizer hyperparameters (lr, momentum, etc...)

Below we will see illustrative examples of the use of the most common optimizers: SGD, RMSProp, Adagrad, Adam, Adadelta.

Let's start with Stochastic Gradient Descent (SGD)

In [ ]:
# We import the necessary libraries
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch import nn
from torch import optim
import torch.nn.functional as F

# We define the transformations for the data sets
transform = transforms.Compose([
    # We transform the images into tensors
    transforms.ToTensor(),
    # We normalize the tensors with mean 0.5 and standard deviation 0.5
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  
])

# We load the CIFAR-10 data sets
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

# We define a simple convolutional network
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)  # 3 RGB input channels, 6 output channels, 5x5 kernel
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)  # 10 output classes for CIFAR-10

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = Net()

# We specify the loss function and the optimizer
# In CrossEntropyLoss, the best value of the loss function is 0
criterion = nn.CrossEntropyLoss()
# SGD: Stochastic Gradient Descent
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, foreach=False, nesterov=False)

# Finally, we train the model
# For 10 eras
for epoch in range(10):
    running_loss = 0.0
    # For each batch of data
    for i, data in enumerate(trainloader, 0):
        # We get the inputs and batch labels from the training set
        inputs, labels = data
        # We reset the gradients
        optimizer.zero_grad()
        # We make a forward pass
        outputs = model(inputs)
        # We calculate the loss
        loss = criterion(outputs, labels)
        # We make a backward pass
        loss.backward()
        # We update the parameters
        optimizer.step()
        # We print statistics
        running_loss += loss.item()
        
print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")


Files already downloaded and verified
Files already downloaded and verified
Época 1, Pérdida: 1.787949834485798
Época 2, Pérdida: 1.4160639706170162
Época 3, Pérdida: 1.2677134099366414
Época 4, Pérdida: 1.175192718691838
Época 5, Pérdida: 1.1008047457698666
Época 6, Pérdida: 1.0454528957529141
Época 7, Pérdida: 0.9847628784454082
Época 8, Pérdida: 0.9480513310645853


In the cell above, we can make different executions by changing the values ​​of the optimizer construction cell parameters:

optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, foreach=False, nesterov=False)

Next, we are going to do the same, but changing the SGD optimizer to Adagrad.

In [ ]:
# We import the necessary libraries
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch import nn
from torch import optim
import torch.nn.functional as F
# We define the transformations for the data sets
transform = transforms.Compose([
    # We transform the images into tensors
    transforms.ToTensor(),
    # We normalize the tensors with mean 0.5 and standard deviation 0.5
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  
])

# We load the CIFAR-10 data sets
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False)

# We define a simple convolutional network
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)  # 3 RGB input channels, 6 output channels, 5x5 kernel
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)  # 10 output classes for CIFAR-10

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = Net()

# We specify the loss function and the optimizer
# In CrossEntropyLoss, the best value of the loss function is 0
criterion = nn.CrossEntropyLoss()

In [ ]:

# Adagrad
optimizer = optim.Adagrad(model.parameters(), lr=0.01, lr_decay=0, weight_decay=0, initial_accumulator_value=0, eps=1e-10)

# Finally, we train the model
# For 10 eras
for epoch in range(10):
    running_loss = 0.0
    # For each batch of data
    for i, data in enumerate(trainloader, 0):
        # We get the inputs and batch labels from the training set
        inputs, labels = data
        # We reset the gradients
        optimizer.zero_grad()
        # We make a forward pass
        outputs = model(inputs)
        # We calculate the loss
        loss = criterion(outputs, labels)
        # We make a backward pass
        loss.backward()
        # We update the parameters
        optimizer.step()
        # We print statistics
        running_loss += loss.item()
        
print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")

We observe the difference in execution time, and in the value of the loss function reached after 10 epochs.

We can make changes to the optimizer constructor parameters:

optimizer = optim.Adagrad(model.parameters(), lr=0.01, lr_decay=0, weight_decay=0, initial_accumulator_value=0, eps=1e-10)

and see what influence these changes have on the execution time and the final precision achieved after the 10 epochs.

Now, let's successively test the RMSProp and Adam optimizers with the same example.

In [ ]:
# RMSprop
optimizer = optim.RMSprop(model.parameters(), lr=0.01, alpha=0.99, eps=1e-08, weight_decay=0, momentum=0, centered=False)
# Finally, we train the model
# For 10 eras
for epoch in range(10):
    running_loss = 0.0
    # For each batch of data
    for i, data in enumerate(trainloader, 0):
        # We get the inputs and batch labels from the training set
        inputs, labels = data
        # We reset the gradients
        optimizer.zero_grad()
        # We make a forward pass
        outputs = model(inputs)
        # We calculate the loss
        loss = criterion(outputs, labels)
        # We make a backward pass
        loss.backward()
        # We update the parameters
        optimizer.step()
        # We print statistics
        running_loss += loss.item()
        
print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")

In [ ]:
# Adam
optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999), eps=1e-08, weight_decay=0, amsgrad=False)
# Finally, we train the model
# For 10 eras
for epoch in range(10):
    running_loss = 0.0
    # For each batch of data
    for i, data in enumerate(trainloader, 0):
        # We get the inputs and batch labels from the training set
        inputs, labels = data
        # We reset the gradients
        optimizer.zero_grad()
        # We make a forward pass
        outputs = model(inputs)
        # We calculate the loss
        loss = criterion(outputs, labels)
        # We make a backward pass
        loss.backward()
        # We update the parameters
        optimizer.step()
        # We print statistics
        running_loss += loss.item()
        
print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")

In both cases above, we can do the same hyperparameter variation tests as we did with the other two optimizers, and observe the effect they have.

## Learning Rate Adjustment

In the previous examples, we have set the *Learning Rate* (*lr*) at the time of creation of the optimizer, as a parameter of the same. It is true that some of these algorithms adapt the *lr* dynamically, based on statistical data from the training, but there is also another mechanism that allows the learning rate to be adapted dynamically, from outside the optimizer. Both mechanisms can be used together, but this could produce unexpected effects. It would be usual to use these adaptation mechanisms in conjunction with optimizers such as SGD that do not incorporate a mechanism of this type.

Let's explore its usage by retrieving the SGD version of our example. In it, we use one of these schedulers (StepLR). For information, we print the *lr* at the end of each epoch, so that we can observe the updates performed.


In [ ]:

from torch.optim import lr_scheduler
# SGD: Stochastic Gradient Descent
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, foreach=False, nesterov=False)
# We created a learning rate scheduler
# Every 7 epochs, the learning rate will be multiplied by 0.1
scheduler = lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)

# Finally, we train the model
# For 10 eras
for epoch in range(10):
    running_loss = 0.0
    # For each batch of data
    for i, data in enumerate(trainloader, 0):
        # We get the inputs and batch labels from the training set
        inputs, labels = data
        # We reset the gradients
        optimizer.zero_grad()
        # We make a forward pass
        outputs = model(inputs)
        # We calculate the loss
        loss = criterion(outputs, labels)
        # We make a backward pass
        loss.backward()
        # We update the parameters using the optimizer step
        optimizer.step()
        # We print statistics
        running_loss += loss.item()
        
print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")
    # We update the lr using the lr planner step
    scheduler.step()
print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}, Learning rate: {scheduler.get_lr()[0]}")



We now try a more sophisticated scheduler: *CosineAnnealingLR*

In [ ]:

from torch.optim import lr_scheduler
# SGD: Stochastic Gradient Descent
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, foreach=False, nesterov=False)
# We created a learning rate scheduler
# Every 7 epochs, the learning rate will be multiplied by 0.1
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=2, eta_min=0)

# Finally, we train the model
# For 10 eras
for epoch in range(10):
    running_loss = 0.0
    # For each batch of data
    for i, data in enumerate(trainloader, 0):
        # We get the inputs and batch labels from the training set
        inputs, labels = data
        # We reset the gradients
        optimizer.zero_grad()
        # We make a forward pass
        outputs = model(inputs)
        # We calculate the loss
        loss = criterion(outputs, labels)
        # We make a backward pass
        loss.backward()
        # We update the parameters using the optimizer step
        optimizer.step()
        # We print statistics
        running_loss += loss.item()
        
print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")
    # We update the lr using the lr planner step
    scheduler.step()
print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}, Learning rate: {scheduler.get_lr()[0]}")



## Weight averaging

Stochastic Weight Averaging (SWA) improves the generalization capacity of the model by averaging its weights over a given number of iterations.

The average is done based on several points in the optimization path. We will see a commented example of the use of this technique in the example we have used throughout this notebook.

In [ ]:

from torch.optim import lr_scheduler
# SGD: Stochastic Gradient Descent
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, foreach=False, nesterov=False)
# We created a learning rate scheduler
# Every 7 epochs, the learning rate will be multiplied by 0.1
swa_model=torch.optim.swa_utils.AveragedModel(model)
scheduler = lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)
swa_start=5
swa_scheduler = torch.optim.swa_utils.SWALR(optimizer, swa_lr=0.05)

# Finally, we train the model
# For 10 eras
for epoch in range(10):
    running_loss = 0.0
    # For each batch of data
    for i, data in enumerate(trainloader, 0):
        # We get the inputs and batch labels from the training set
        inputs, labels = data
        # We reset the gradients
        optimizer.zero_grad()
        # We make a forward pass
        outputs = model(inputs)
        # We calculate the loss
        loss = criterion(outputs, labels)
        # We make a backward pass
        loss.backward()
        # We update the parameters using the optimizer step
        optimizer.step()
        # We print statistics
        running_loss += loss.item()
        
print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")
    # We update the lr using the lr planner step
    if epoch>swa_start:
        swa_model.update_parameters(model)
        swa_scheduler.step()
print(f"SWA: Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}, Learning rate: {swa_scheduler.get_lr()[0]}")

    else:
        scheduler.step()
print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}, Learning rate: {scheduler.get_lr()[0]}")

torch.optim.swa_utils.update_bn(trainloader, swa_model)



# Exercise

Now we are going to do an exercise on optimizers. The objective is to test different optimization algorithms on the same example and compare their performance.
The code is prepared to train the same network with three different optimization algorithms: SGD, Adam and RMSProp. The missing part of the code is the definition of the use of the Adam and RMSProp optimizers, with the SGD code provided as a reference.

At the end of the three training sessions, a graph is printed comparing the evolution of the loss function in the training, when each of the 3 optimizers are used.



In [ ]:
# Import libraries
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

# Load CIFAR-10 dataset
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=2)

# Define the network architecture
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Define the loss function
loss_func = nn.CrossEntropyLoss()

# Define the training function
def train(net, trainloader, optimizer, num_epochs=2):
    loss_values = []
    for epoch in range(num_epochs):  # loop over the dataset multiple times
        running_loss = 0.0
        for i, data in enumerate(trainloader, 0):
            # get the inputs; data is a list of [inputs, labels]
            inputs, labels = data

            # zero the parameter gradients
            optimizer.zero_grad()

            # forward + backward + optimize
            outputs = net(inputs)
            loss = loss_func(outputs, labels)
            loss.backward()
            optimizer.step()

            # save losses to plot later
            running_loss += loss.item()
            if i % 2000 == 1999:  # print every 2000 mini-batches
                loss_values.append(running_loss / 2000)
                running_loss = 0.0
    return loss_values

# SGD
net_SGD = Net()
optimizer_SGD = optim.SGD(net_SGD.parameters(), lr=0.001, momentum=0.9)
loss_SGD = train(net_SGD, trainloader, optimizer_SGD)

# Adam
net_Adam = Net()
# Write your code here
# None


# RMSprop
net_RMSprop = Net()
# Write your code here
# None

# Plotting
plt.figure(figsize=(12, 8))
plt.plot(loss_SGD, label='SGD')
plt.plot(loss_Adam, label='Adam')
plt.plot(loss_RMSprop, label='RMSprop')
plt.xlabel('Epochs (x2000 mini-batches)')
plt.ylabel('Loss')
plt.title('Loss Evolution with Different Optimizers')
plt.legend()
plt.show()
